# Initialization

In [ ]:
import serial
import numpy as np

# To communicate with the pico, you will need to determine the port the pico
# is plugged into. On Windows, you can open device manager and look at the
# 'Ports (COM & LPT)' dropdown, where the pico will show up as 'USB Serial Device'
PICO_PORT = "/dev/tty.usbmodem1101"

MHZ = 1000000


class PrawnBlasterClient:
    """
    Helper class to setup functions for pico interface

    Attributes:
        port (str): port pico is assigned to (e.g., 'COM4')
        baudrate (int): speed of data transmission to pico (default=115200 bps)
        timout (float): read timeout in seconds
        debug (bool): activate extra debug messages over serial (only 'on' or 'off')
        ser (): TODO
    """

    def __init__(self, port=PICO_PORT, baudrate=115200, timeout=1, debug=True):
        """Setup and initialize PrawnBlaster"""
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.debug = debug
        self.ser = None

    def open(self, ):
        """Open serial port for pico communication"""
        if self.ser and self.ser.is_open:
            return
        self.ser = serial.Serial(
            port=self.port, baudrate=self.baudrate, timeout=self.timeout)
        if self.debug:
            print(f"Opened serial connection on {self.port}")

    def close(self):
        """Close serial port for pico communication"""
        if self.ser and self.ser.is_open:
            self.ser.close()
            if self.debug:
                print(f"Closed serial connection on {self.port}")

    def response(self):  # TODO: This does the same as end of send_command
        """Retrieve the response from the pico"""
        if self.ser and self.ser.is_open:
            self.ser.readline()

    def send_command(self, cmd: str, read_bytes=200) -> str:
        """Send a command string and return the response"""
        if not self.ser or not self.ser.is_open:
            self.open()
        if self.debug:
            print(f"> {cmd.strip()}")
        if cmd[-2:] != '\r\n':
            cmd += '\r\n'

        response = ''
        try:
            self.ser.write(cmd.encode())
            response = self.ser.read(read_bytes).decode(errors="ignore")
            if self.debug:
                print(f"< {response.strip()}\n")
        except Exception as e:
            print("Encountered Error: ", e)
        return response

### Test Serial Communication with the Pico

In [ ]:
pb = PrawnBlasterClient()

# debug mode within serial communication on or off:
debugmode = "on"

# set number of psuedoclocks
numofclocks = 4

pb.open()
pb.send_command('status\r\n')
pb.send_command(f'debug {debugmode}\r\n')
pb.send_command(f'setnumpseudoclocks {numofclocks}\r\n')
pb.close()

### Set Frequency

In [ ]:
# 0 - internal reference frequency; 1 - external reference frequency
mode = 0

# set frequency in MHZ
frequencyMHZ = 200
freqHZ = frequencyMHZ * MHZ

# change frequency, and check frequency/voltage
pb.open()
pb.send_command(f'setclock {mode} {freqHZ}\r\n')
pb.send_command('getfreqs\r\n')
pb.send_command('getvolts\r\n')
pb.close()

### Basic Testing: Go High and Go Low

In [ ]:
# Testing Pico Logic
pb.open()
pb.send_command(cmd="go high 0/r/n", read_bytes=10000)
pb.close()

In [ ]:
pb.open()
pb.send_command(cmd="go low 0/r/n", read_bytes=10000)
pb.close()

# Tests

### Test 1: Single Frequency

In [ ]:
# Change the parameters below as needed
# Set Psuedoclock (0 - 3)
psuedoclock = 0

# Beginning Address (0 indexed)
baddr = 0

# Ending Address
eaddr = 1

# Stop Address (Set Equal to the last Ending Address)
saddr = 1

# Set Half-period (5 - 2^32)
halfperiod = 30

# Set Repetitions (0 - 2^32)
reps = 10000

pb.open()
for i in range(baddr, eaddr):
    pb.send_command(f"set {psuedoclock} {i} {halfperiod} {reps}\n\r")
pb.send_command(f"set {psuedoclock} {saddr} 0 0\r\n")
pb.close()

In [ ]:
# run instructions above
pb.open()
pb.send_command(cmd='start\r\n', read_bytes=10000)
pb.close()

### Test 2: Multiple Frequencies without Waits

Only run either 1, 2, or 3, as they will overwrite each other. Each one tests the pico in a different way.

#### 1. Full pulse sequency, no Waits

In [ ]:
instructions_noWaits = np.array([
    [100, 1],
    [6, 1],
    [101, 3],
    [6, 5],
    [7, 1],
    [6, 1],
    [7, 1],
    [6, 1],
    [7, 1],
    [6, 1],
    [7, 1],
    [6, 1],
    [100, 5],
    [12, 3],
    [52, 4],
    [53, 4],
    [100000000, 1],
    [6, 1],
    [0, 0]
])

instructions_noWaits_Lengths = np.prod(instructions_noWaits, axis=1)*10*2
instructions_noWaits_TotalLength = np.sum(instructions_noWaits_Lengths)
[_, instructions_noWaits_pulses] = np.sum(instructions_noWaits, axis=0)
print(f'Total length: {instructions_noWaits_TotalLength:d} ns')
print(
    f'Total number of pulses: {instructions_noWaits_pulses:d}, Edges: {instructions_noWaits_pulses*2:d}')
print(instructions_noWaits_Lengths)

#### 2. Mid Length pulses only

In [ ]:
base = 90
instructions_noWaits = np.array([
    [base, 1],
    [base+1, 3],
    [base, 5],
    [base+20, 3],
    [base+21, 1],
    [base+20, 1],
    [base+21, 1],
    [base+20, 1],
    [base+21, 1],
    [base+25, 4],
    [base+28, 4],
    [0, 0]
])

instructions_noWaits_Lengths = np.prod(instructions_noWaits, axis=1)*10*2
instructions_noWaits_TotalLength = np.sum(instructions_noWaits_Lengths)
[_, instructions_noWaits_pulses] = np.sum(instructions_noWaits, axis=0)
print(f'Total length: {instructions_noWaits_TotalLength:d} ns')
print(
    f'Total number of pulses: {instructions_noWaits_pulses:d}, Edges: {instructions_noWaits_pulses*2:d}')
print(instructions_noWaits_Lengths)

#### 3. Increasing Length Pulses

In [ ]:
base = 1000000
instructions_noWaits = np.array([
    [base, 1],
    [base*2, 1],
    [base*5, 1],
    [base*10, 1],
    [base*20, 1],
    [base*50, 1],
    [base*100, 1],
    [0, 0]
])

instructions_noWaits_Lengths = np.prod(instructions_noWaits, axis=1)*10*2
instructions_noWaits_TotalLength = np.sum(
    instructions_noWaits_Lengths, dtype=float)
[_, instructions_noWaits_pulses] = np.sum(instructions_noWaits, axis=0)
print(f'Total length: {instructions_noWaits_TotalLength:e} ns')
print(
    f'Total number of pulses: {instructions_noWaits_pulses:d}, Edges: {instructions_noWaits_pulses*2:d}')
print(instructions_noWaits_Lengths)

#### Send command sequence to pico

In [ ]:
# double check that you are sending the correct instructions
for i, line in enumerate(instructions_noWaits):
    print(f'set 0 {i:d} {line[0]:d} {line[1]:d}\r\n'.encode())

In [ ]:
pb.open()

# do I need this full section? user could look at whether there is an ok after every command
for i, line in enumerate(instructions_noWaits):
    pb.send_command(f'set 0 {i:d} {line[0]:d} {line[1]:d}\r\n')
    # if pb.response() == b'ok\r\n':
    #     continue
    # else:
    #     print(f'Error on line {i:d}!')
    #     break

pb.send_command(cmd='start\r\n', read_bytes=10000)
pb.close()